# 09 — Capstone: A Tiny Causal Transformer Language Model

    **Companion chapter:** `09-key-takeaways-and-references.md`

    ## Learning goals

    - Combine embeddings, positions, causal masking, encoder-style blocks, and logits.
- Train a small next-token model end to end.
- Calculate loss and approximate perplexity.
- Generate tokens autoregressively.
- Identify the changes needed for a real Persian corpus experiment.

    ## How to use this notebook

    Run the cells from top to bottom. Read the comments, change small values, and
    rerun the cell. Every notebook ends with practice prompts that can become
    GitHub issues, exercises, or discussion questions.

In [1]:
from __future__ import annotations

import math
import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## 1. Capstone goal

We build a tiny **causal Transformer language model**. For simplicity, PyTorch's
`TransformerEncoder` blocks are used with a causal mask. Architecturally, this
behaves like a small decoder-only language model:

`tokens + positions → masked self-attention blocks → vocabulary logits`

The corpus is intentionally tiny and is only for learning the pipeline.

In [2]:
corpus = [
    "من شعر فارسی را دوست دارم",
    "من کتاب فارسی را می خوانم",
    "سارا شعر نو را دوست دارد",
    "علی کتاب زبان را می خواند",
    "مدل زبانی واژه بعدی را پیش بینی می کند",
    "توجه رابطه میان واژه ها را یاد می گیرد",
]

repetitions = 25
sentences = corpus * repetitions

special = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]
words = sorted({word for sentence in sentences for word in sentence.split()})
itos = special + words
stoi = {token: index for index, token in enumerate(itos)}

PAD_ID = stoi["<PAD>"]
BOS_ID = stoi["<BOS>"]
EOS_ID = stoi["<EOS>"]
UNK_ID = stoi["<UNK>"]

stream = []
for sentence in sentences:
    stream.extend([BOS_ID])
    stream.extend(stoi.get(token, UNK_ID) for token in sentence.split())
    stream.extend([EOS_ID])

print("Vocabulary size:", len(itos))
print("Training tokens:", len(stream))

Vocabulary size: 32
Training tokens: 1350


## 2. Create fixed-length next-token examples

In [3]:
block_size = 10

x_examples = []
y_examples = []

for start in range(len(stream) - block_size):
    window = stream[start : start + block_size + 1]
    x_examples.append(window[:-1])
    y_examples.append(window[1:])

X = torch.tensor(x_examples, dtype=torch.long)
Y = torch.tensor(y_examples, dtype=torch.long)

print("X:", tuple(X.shape))
print("Y:", tuple(Y.shape))
print("First input:", [itos[index] for index in X[0].tolist()])
print("First target:", [itos[index] for index in Y[0].tolist()])

X: (1340, 10)
Y: (1340, 10)
First input: ['<BOS>', 'من', 'شعر', 'فارسی', 'را', 'دوست', 'دارم', '<EOS>', '<BOS>', 'من']
First target: ['من', 'شعر', 'فارسی', 'را', 'دوست', 'دارم', '<EOS>', '<BOS>', 'من', 'کتاب']


## 3. Define the model

In [4]:
class TinyTransformerLM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        block_size: int,
        d_model: int = 48,
        num_heads: int = 4,
        num_layers: int = 2,
        d_ff: int = 96,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.vocabulary_projection = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        batch_size, length = token_ids.shape
        if length > self.block_size:
            raise ValueError("Input is longer than block_size.")

        positions = torch.arange(length, device=token_ids.device)
        x = self.token_embedding(token_ids)
        x = x + self.position_embedding(positions).unsqueeze(0)

        mask = torch.triu(
            torch.ones(length, length, dtype=torch.bool, device=token_ids.device),
            diagonal=1,
        )
        x = self.blocks(x, mask=mask, is_causal=True)
        x = self.final_norm(x)
        return self.vocabulary_projection(x)


model = TinyTransformerLM(
    vocab_size=len(itos),
    block_size=block_size,
).to(device)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Parameters: {parameter_count:,}")

test_logits = model(X[:2].to(device))
print("Logits:", tuple(test_logits.shape))

Parameters: 41,600
Logits: (2, 10, 32)


/tmp/ipykernel_1008/733177221.py:25: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(


## 4. Train with random mini-batches

This tiny corpus is repeated so the model can overfit quickly. Do not interpret
the generated text as evidence of general language ability.

In [5]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
loss_function = nn.CrossEntropyLoss()

steps = 120
batch_size = 32
loss_history = []

model.train()
for step in range(steps):
    indices = torch.randint(0, len(X), (batch_size,))
    x_batch = X[indices].to(device)
    y_batch = Y[indices].to(device)

    optimizer.zero_grad()
    logits = model(x_batch)
    loss = loss_function(logits.reshape(-1, len(itos)), y_batch.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    loss_history.append(loss.item())

    if (step + 1) % 30 == 0:
        print(f"step {step + 1:3d} | loss {loss.item():.3f}")

plt.plot(loss_history)
plt.xlabel("Optimization step")
plt.ylabel("Cross-entropy loss")
plt.title("Tiny causal Transformer training")
plt.show()

step  30 | loss 0.704
step  60 | loss 0.148


step  90 | loss 0.112
step 120 | loss 0.127


/tmp/ipykernel_1008/3356034157.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Evaluate approximate perplexity

In [6]:
model.eval()
with torch.no_grad():
    evaluation_logits = model(X[:256].to(device))
    evaluation_loss = loss_function(
        evaluation_logits.reshape(-1, len(itos)),
        Y[:256].to(device).reshape(-1),
    )
    perplexity = math.exp(evaluation_loss.item())

print("Evaluation loss:", round(evaluation_loss.item(), 3))
print("Approximate perplexity:", round(perplexity, 3))

Evaluation loss: 0.118
Approximate perplexity: 1.125


## 6. Greedy generation

In [7]:
@torch.no_grad()
def generate(
    model: nn.Module,
    prefix: list[str],
    max_new_tokens: int = 12,
    temperature: float = 1.0,
    sample: bool = False,
) -> list[str]:
    model.eval()
    generated = [BOS_ID]
    generated.extend(stoi.get(token, UNK_ID) for token in prefix)

    for _ in range(max_new_tokens):
        context = generated[-block_size:]
        x = torch.tensor([context], dtype=torch.long, device=device)
        logits = model(x)[0, -1] / temperature
        probabilities = torch.softmax(logits, dim=-1)

        if sample:
            next_id = int(torch.multinomial(probabilities, num_samples=1))
        else:
            next_id = int(probabilities.argmax())

        if next_id == EOS_ID:
            break
        generated.append(next_id)

    return [itos[index] for index in generated[1:]]


print("Greedy:", " ".join(generate(model, ["من"])))
print("Sampled:", " ".join(generate(model, ["مدل"], temperature=0.8, sample=True)))

Greedy: من کتاب فارسی را می خوانم
Sampled: مدل زبانی واژه بعدی را پیش بینی می کند


## 7. Optional checkpoint

Uncomment the following line to save weights inside your cloned repository.

```python
torch.save(
    {"model_state": model.state_dict(), "stoi": stoi, "itos": itos},
    "tiny_transformer_lm.pt",
)
```

## Capstone extensions

1. Replace the toy corpus with a normalized Persian corpus split into train,
   validation, and test sets.
2. Replace whitespace tokens with subword tokenization.
3. Add padding-aware batches instead of a continuous token stream.
4. Compare learned and sinusoidal positional encodings.
5. Log validation perplexity and stop training when it worsens.
6. Expose attention matrices for linguistic analysis.
7. Compare the tiny Transformer against the GRU from Notebook 01.
8. Add a model card documenting data, limitations, and intended use.